# Piloto 2026 — validación de aromas alineada con el criterio de decisión

## tl;dr

**Veredicto:** `NO_VALID_MODEL`.  
**Modelo seleccionado:** `None`.

Esta iteración no agrega parámetros: compara captura constante y dependiente de etanol usando una ponderación homoscedástica por dominio. El objetivo de calibración queda así alineado con el NRMSE usado en el gate.

## Caveats predefinidos

- La ponderación por media del dominio es una función de decisión, no una estimación de error analítico.
- La eficiencia sigue siendo un operador efectivo de recuperación; no una propiedad termodinámica medida.
- El benchmark de réplicas cuantifica el piso experimental de predicción.
- El análisis es retrospectivo y post hoc; cualquier `PASS` requiere una campaña confirmatoria.

In [ ]:
from pathlib import Path
import os
import sys
from IPython.display import display, Image

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'fermentation_model').exists():
    ROOT = ROOT.parent
if not (ROOT / 'fermentation_model').exists():
    raise RuntimeError('Execute from the repository or a descendant directory')
sys.path.insert(0, str(ROOT / 'fermentation_model'))
from pilot_2026 import run_aroma_decision_aligned_validation_2026 as analysis
result = analysis.load_results() if os.environ.get('PILOT_AROMA_REUSE_RESULTS') == '1' else analysis.run_analysis()
print('Veredicto:', result['gate']['verdict'])
print('Seleccionado:', result['gate']['selected_model'])

## Reproducibilidad experimental

In [ ]:
display(result['repeatability'].round(4))
display(result['repeatability_detail'].round(3))
display(Image(filename=analysis.FIGURE_DIR / '01_sample_anomaly_audit.png'))

## Validación leave-one-run-out

In [ ]:
display(result['metrics'].round(4))
display(result['comparison'].round(4))
display(Image(filename=analysis.FIGURE_DIR / '02_capture_efficiency_loro_nrmse.png'))
display(Image(filename=analysis.FIGURE_DIR / '07_capture_efficiency_by_fold.png'))

## Forzantes y curvas

In [ ]:
display(Image(filename=analysis.FIGURE_DIR / '03_rco2_temperature_pulse_drivers.png'))
for species in analysis.ethanol.capture.base.SPECIES_LABELS:
    display(Image(filename=analysis.FIGURE_DIR / f'04_liquid_{species}.png'))
    display(Image(filename=analysis.FIGURE_DIR / f'05_condensate_{species}.png'))

## Estabilidad paramétrica

In [ ]:
display(result['stability'].round(5))
display(result['parameters'].round(6))
display(result['fit_validation'].round(5))

## Takeaways

- El gate compara la extensión dependiente de etanol contra la captura constante bajo la misma función de decisión.
- Si el error se aproxima al benchmark de réplicas, más complejidad no puede justificarse sin mejorar el protocolo de medición.
- El ensayo confirmatorio debe incluir estándar gaseoso, concentración en gas de salida, volumen/%EtOH por MIX y duplicados de condensado.

In [ ]:
assert result['gate']['verdict'] in {'PASS', 'NO_VALID_MODEL'}
assert len(result['figures']) == 8
assert result['fit_validation']['maximum_relative_mass_balance_error'].max() <= 1e-8
print('Notebook ejecutado sin errores; figuras:', len(result['figures']))